# Comparing solutions: which one is more efficient?

This notebook compares two (or more) ways of solving the same exercise from [mnist_numpy.ipynb](mnist_numpy.ipynb) and measures which one is faster. Every comparison follows the same steps:

1. Prepare the data **once**, outside what is timed.
2. Check that every version gives **the same result**: comparing the speed of two solutions that do different things is pointless.
3. Time each version with `timeit`.
4. Write a conclusion: which one is better and why.

## Imports and data

- `timeit` runs a function many times and returns the total time in seconds. Repeating the measurement reduces the noise of a single run.
- `numpy` is the library the exercises are solved with.
- `load_mnist` is the same function as in the exercises notebook: it downloads MNIST into the `data/` folder the first time and reads it from there afterwards.

The variables are the same as in the exercises:

| Variable | Contents | Shape |
|---|---|---|
| `X_train` | training images, values from 0 to 255 (`uint8`) | `(60000, 28, 28)` |
| `y_train` | label (digit 0–9) of each training image | `(60000,)` |
| `X_test` | test images | `(10000, 28, 28)` |
| `y_test` | test labels | `(10000,)` |

In [1]:
import hashlib
import timeit
import urllib.request
from pathlib import Path

import numpy as np

MNIST_URL = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
MNIST_SHA256 = "731c5ac602752760c8e48fbffcf8c3b850d9dc2a2aedcf2cc48468fc17b673d1"

type ImagesAndLabels = tuple[np.ndarray, np.ndarray]


def load_mnist(cache_dir: Path = Path("data")) -> tuple[ImagesAndLabels, ImagesAndLabels]:
    """Download MNIST into `cache_dir` once, check it, and return its (train, test) splits."""
    cache_dir.mkdir(exist_ok=True)
    path = cache_dir / "mnist.npz"
    if not path.exists():
        urllib.request.urlretrieve(MNIST_URL, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != MNIST_SHA256:
        path.unlink()
        raise ValueError("The download is corrupted and has been deleted: run the cell again")
    with np.load(path) as mnist:
        return (mnist["x_train"], mnist["y_train"]), (mnist["x_test"], mnist["y_test"])


(X_train, y_train), (X_test, y_test) = load_mnist()
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

### Helper function `compare`

To avoid repeating the same code in every comparison, `compare` takes the versions as keyword arguments (`name=lambda: ...`). It first checks that they all return the same thing and then times each one. It prints the average time **per call**, which is easier to read than the total, and how many times slower each version is than the fastest one.

In [2]:
def compare(number: int = 1000, **versions) -> None:
    """Check that every version returns the same result, then time each one `number` times."""
    results = {name: f() for name, f in versions.items()}
    reference_name, reference = next(iter(results.items()))
    for name, result in results.items():
        assert np.array_equal(result, reference), f"{name} does not return the same as {reference_name}"

    times = {name: timeit.timeit(f, number=number) / number for name, f in versions.items()}
    fastest = min(times.values())
    for name, t in sorted(times.items(), key=lambda item: item[1]):
        print(f"{name:>20}: {t * 1e6:10.2f} µs per call  (x{t / fastest:.2f})")

## Example: are all the pixels at one position 0?

We want to know whether pixel `(27, 27)` (the bottom-right corner) is 0 in **every** training image. There are two equivalent ways to ask:

- `np.all(pixels == 0)`: "are they all zero?".
- `not np.any(pixels != 0)`: "is there none that is not zero?".

The pixels are extracted once, outside the measurement, so that only the check is timed.

In [3]:
# Extract the pixels once
pixels = X_train[:, 27, 27]

compare(
    number=10_000,
    all_equal_zero=lambda: np.all(pixels == 0),
    not_any_nonzero=lambda: not np.any(pixels != 0),
)

      all_equal_zero:     110.78 µs per call  (x1.00)
     not_any_nonzero:     115.61 µs per call  (x1.04)


**Conclusion:** _write here which one is faster and why._

Hint: both versions first build a full boolean array (`pixels == 0` or `pixels != 0`) and then scan it, so they do practically the same work. If the difference you see is small and changes when you rerun the cell, it is noise, and the better option is the easier one to read.

---

## Template

Copy the next three cells (title, code and conclusion) for every new comparison and fill in the blanks.

## Exercise N: _title_

**What is compared:** _describe what the exercise computes and how the versions differ._

- `version_a`: _..._
- `version_b`: _..._

In [4]:
# Data prepared once (not timed)
data = X_train

compare(
    number=100,
    version_a=lambda: data.reshape(len(data), -1),
    version_b=lambda: data.reshape(len(data), 28 * 28),
)

           version_a:       0.30 µs per call  (x1.00)
           version_b:       0.36 µs per call  (x1.22)


**Conclusion:** _which one is better and why (speed, memory, readability)._